# core

> ledger.csv backend — pure logic and atomic CSV I/O (see `DESIGN.md` §3). The web layer lives in `01_app.ipynb`.

In [ ]:
#| default_exp core

In [ ]:
#| export
import os
from datetime import date, timedelta
from pathlib import Path

import pandas as pd

In [ ]:
#| hide
import tempfile

## `save_full`

In [ ]:
#| export
def save_full(
    df:pd.DataFrame,  # ledger frame — index=date, columns=item names, values=bool
    path:Path):  # live CSV; a .tmp sibling in the same dir is used for the swap
    "Atomic write: a mid-write crash must never corrupt the live file."
    path.parent.mkdir(exist_ok=True)
    tmp = path.with_suffix(".tmp")
    df.to_csv(tmp)
    os.replace(tmp, path)

In [ ]:
with tempfile.TemporaryDirectory() as d:
    p = Path(d) / "ledger.csv"
    df = pd.DataFrame({"読書": [True, False]},
                      index=pd.Index([date(2026, 8, 23), date(2026, 8, 24)], name="date"))
    save_full(df, p)
    assert p.exists() and not p.with_suffix(".tmp").exists()
    back = pd.read_csv(p, index_col=0)
    assert list(back.columns) == ["読書"]
    assert back["読書"].tolist() == [True, False]

## `load_full`

In [ ]:
#| export
def load_full(
    path:Path,  # ledger CSV location
    default_items:list,  # columns for first-run auto-create
    today:date,  # index row for first-run auto-create
) -> pd.DataFrame:
    "Full history; blank cells read as False; auto-creates the CSV on first run."
    if not path.exists():
        df = pd.DataFrame(False, index=pd.Index([today], name="date"),
                          columns=default_items)
        save_full(df, path)
        return df
    df = pd.read_csv(path, index_col=0)
    df = df.fillna(False).astype(bool)
    df.index = pd.Index([date.fromisoformat(d) for d in df.index], name="date")
    return df

In [ ]:
with tempfile.TemporaryDirectory() as d:
    p = Path(d) / "ledger.csv"
    t = date(2026, 8, 24)

    # first run: auto-create, everything False
    df = load_full(p, ["読書", "運動"], t)
    assert p.exists()
    assert list(df.columns) == ["読書", "運動"]
    assert list(df.index) == [t]
    assert df.dtypes.eq(bool).all() and not df.values.any()

    # blank cells (e.g. a hand-added column) read as False, never True
    p.write_text("date,読書,運動\n2026-08-23,True,\n2026-08-24,,False\n")
    df = load_full(p, [], t)
    assert df.dtypes.eq(bool).all()
    assert df.at[date(2026, 8, 23), "読書"] == True
    assert df.at[date(2026, 8, 23), "運動"] == False
    assert df.at[date(2026, 8, 24), "読書"] == False
    assert df.at[date(2026, 8, 24), "運動"] == False

## `window`

In [ ]:
#| export
def window(
    full:pd.DataFrame,  # full history from `load_full`
    today:date,  # last day of the window
    days:int=10,  # window length
) -> pd.DataFrame:
    "Trailing `days`-day view ending today; dates missing from history are False."
    idx = [today - timedelta(days=i) for i in range(days - 1, -1, -1)]
    return full.reindex(pd.Index(idx, name="date"), fill_value=False)

In [ ]:
t = date(2026, 8, 24)
full = pd.DataFrame({"読書": [True]},
                    index=pd.Index([date(2026, 8, 20)], name="date")).astype(bool)
w = window(full, t, 10)
assert len(w) == 10
assert list(w.index)[0] == date(2026, 8, 15) and list(w.index)[-1] == t
assert w.at[date(2026, 8, 20), "読書"] == True
assert w.at[t, "読書"] == False  # date missing from history -> False
assert w.dtypes.eq(bool).all()

## Item management

Items live only in the CSV header; an archived item is a column whose name starts with `~` (DESIGN.md §5). Everything here is a pure `DataFrame -> DataFrame` transform — I/O stays in `save_full`/`load_full`.

In [ ]:
#| hide
def raises(exc, fn, *args):
    "Assert that `fn(*args)` raises `exc`."
    try: fn(*args)
    except exc: return
    raise AssertionError(f"{exc.__name__} not raised")

In [ ]:
#| export
ARCHIVE_PREFIX = "~"  # leading marker on the column name of an archived item

## `is_archived`

In [ ]:
#| export
def is_archived(
    name:str,  # column name as stored in the CSV header
) -> bool:
    "True when the column name carries the archive marker."
    return name.startswith(ARCHIVE_PREFIX)

In [ ]:
assert is_archived("~読書") == True
assert is_archived("読書") == False
assert is_archived("読~書") == False  # only a leading marker counts

## `display_name`

In [ ]:
#| export
def display_name(
    name:str,  # column name as stored in the CSV header
) -> str:
    "Item name without the archive marker."
    return name[len(ARCHIVE_PREFIX):] if is_archived(name) else name

In [ ]:
assert display_name("~読書") == "読書"
assert display_name("読書") == "読書"

## `validate_name`

In [ ]:
#| export
def validate_name(
    name:str,  # candidate item name (display form, no marker)
    existing:list,  # current column names, archived ones included
):
    "Raise ValueError when `name` would corrupt the CSV, break routing, or collide."
    if not name:
        raise ValueError("empty name")
    if is_archived(name):
        raise ValueError(f"leading {ARCHIVE_PREFIX!r} is reserved for archived items")
    bad = set(',"/\n\r') & set(name)  # CSV parsing / path routing (DESIGN.md §5)
    if bad:
        raise ValueError(f"forbidden characters: {''.join(sorted(bad))!r}")
    if name in (display_name(c) for c in existing):
        raise ValueError(f"duplicate: {name}")

In [ ]:
validate_name("新項目", ["読書", "~運動"])  # ok
validate_name("A?B #1 %", ["読書"])  # URL metacharacters are fine (percent-encoded by toggle_url)
for bad in ["", "~x", "a,b", 'a"b', "a/b", "a\nb"]:
    raises(ValueError, validate_name, bad, [])
raises(ValueError, validate_name, "読書", ["読書"])  # duplicate
raises(ValueError, validate_name, "運動", ["~運動"])  # would collide with the archived item once it is restored

## `ordered_items`

In [ ]:
#| export
def ordered_items(
    df:pd.DataFrame,  # ledger frame
) -> list:
    "Column names for display: active items first, then archived, each in column order."
    cols = list(df.columns)
    return [c for c in cols if not is_archived(c)] + [c for c in cols if is_archived(c)]

In [ ]:
df = pd.DataFrame(columns=["~a", "b", "c", "~d"])
assert ordered_items(df) == ["b", "c", "~a", "~d"]

## `add_item`

In [ ]:
#| export
def add_item(
    df:pd.DataFrame,  # ledger frame
    name:str,  # new item name
) -> pd.DataFrame:
    "New item as the last column; every existing day reads False."
    validate_name(name, list(df.columns))
    out = df.copy()
    out[name] = False
    return out

In [ ]:
idx = pd.Index([date(2026, 8, 24)], name="date")
df = pd.DataFrame({"読書": [True]}, index=idx)
out = add_item(df, "運動")
assert list(out.columns) == ["読書", "運動"]
assert out.at[date(2026, 8, 24), "運動"] == False and out.dtypes.eq(bool).all()
assert list(df.columns) == ["読書"]  # input untouched
raises(ValueError, add_item, df, "読書")

## `rename_item`

In [ ]:
#| export
def _column(
    df:pd.DataFrame,  # ledger frame
    name:str,  # display name (marker optional)
) -> str:
    "Stored column name for a display name; KeyError when absent."
    for c in df.columns:
        if c == name or display_name(c) == name:
            return c
    raise KeyError(name)

def rename_item(
    df:pd.DataFrame,  # ledger frame
    old:str,  # current display name
    new:str,  # new display name
) -> pd.DataFrame:
    "Rename keeping history; an archived item stays archived."
    col = _column(df, old)
    validate_name(new, [c for c in df.columns if c != col])
    if is_archived(col):
        new = ARCHIVE_PREFIX + new
    return df.rename(columns={col: new})

In [ ]:
idx = pd.Index([date(2026, 8, 24)], name="date")
df = pd.DataFrame({"読書": [True], "~運動": [True]}, index=idx)
out = rename_item(df, "読書", "本を読む")
assert list(out.columns) == ["本を読む", "~運動"]
assert out.at[date(2026, 8, 24), "本を読む"] == True  # history kept
out = rename_item(df, "運動", "筋トレ")  # archived: addressed by display name, marker kept
assert list(out.columns) == ["読書", "~筋トレ"]
raises(KeyError, rename_item, df, "無い", "x")
raises(ValueError, rename_item, df, "読書", "運動")  # collides with the archived item

## `set_archived`

In [ ]:
#| export
def set_archived(
    df:pd.DataFrame,  # ledger frame
    name:str,  # display name
    flag:bool,  # True = archive, False = restore
) -> pd.DataFrame:
    "Add or remove the archive marker; no-op when already in that state."
    col = _column(df, name)
    if is_archived(col) == flag:
        return df
    new = ARCHIVE_PREFIX + col if flag else display_name(col)
    return df.rename(columns={col: new})

In [ ]:
idx = pd.Index([date(2026, 8, 24)], name="date")
df = pd.DataFrame({"読書": [True], "~運動": [False]}, index=idx)
assert list(set_archived(df, "読書", True).columns) == ["~読書", "~運動"]
assert list(set_archived(df, "運動", False).columns) == ["読書", "運動"]
assert list(set_archived(df, "読書", False).columns) == ["読書", "~運動"]  # no-op
assert set_archived(df, "読書", True).at[date(2026, 8, 24), "~読書"] == True  # history kept
raises(KeyError, set_archived, df, "無い", True)

## `rate_30d`

In [ ]:
#| export
def rate_30d(
    full:pd.DataFrame,  # full history from `load_full`
    item:str,  # column name
    today:date,  # last day of the 30-day window
) -> float:
    "Fraction of the trailing 30 days (ending today) on which the item was checked."
    return float(window(full, today, 30)[item].mean())

In [ ]:
t = date(2026, 8, 24)
idx = [t - timedelta(days=i) for i in range(40)]
full = pd.DataFrame({"x": [i < 15 for i in range(40)]}, index=pd.Index(idx, name="date"))
assert rate_30d(full, "x", t) == 0.5  # 15 of the last 30 days; days 30-39 are outside the window
assert rate_30d(full, "x", t + timedelta(days=30)) == 0.0
empty = pd.DataFrame({"x": []}, index=pd.Index([], name="date")).astype(bool)
assert rate_30d(empty, "x", t) == 0.0

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()